# Aprendizado de Máquina — Lista prática E3

## NLP + Classificação (Aplicação)

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

Última lista do curso, e a única sobre dados que não chegam em forma de tabela.
Vai aparecer quase tudo: representação, esparsidade, três classificadores, e a
escolha da métrica num problema desbalanceado.

E um resultado que contraria o conselho mais repetido em processamento de texto:

> **trocar contagens por TF-IDF *piora* dois dos três classificadores aqui. A
> transformação certa depende do modelo que vem depois dela.**

Cada lacuna está marcada com `...`. Substitua **cada uma** pela sua resposta e
rode a célula.

---
## 1. Importando os pacotes

In [ ]:
import os

import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots

import sklearn.model_selection as skm
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

import warnings
warnings.filterwarnings("ignore")

---
## Exercício 1 — do texto à matriz esparsa

O `spam.csv` é uma coleção de mensagens de SMS rotuladas. Ele foi exportado com
três colunas vazias sobrando e num encoding antigo — nada disso é acidente do
nosso arquivo, é como dados de texto costumam chegar.

In [ ]:
_nome = "spam.csv"

# procura em dois lugares, sem baixar nada da internet: a pasta deste
# notebook primeiro ou então ../../recursos/dados/
_lugares = [_nome, os.path.join("..", "..", "recursos", "dados", _nome)]
_caminho = next((c for c in _lugares if os.path.exists(c)), None)

if _caminho is None:
    raise FileNotFoundError(
        f"nao encontrei '{_nome}'. Procurei nesta pasta e em "
        "../../recursos/dados/. Ponha o .csv ao lado deste notebook, "
        "ou mude o caminho se for necessário."
    )

mensagens = pd.read_csv(_caminho, encoding=...)           # (a) o arquivo NAO e utf-8
print("colunas do arquivo:", list(mensagens.columns))

mensagens = mensagens.iloc[:, :2]
mensagens.columns = ["rotulo", "texto"]
y = (mensagens["rotulo"] == ...).astype(int).values        # (b)

print(f"n = {len(mensagens)}, spam = {y.mean():.4f} ({int(y.sum())} mensagens)")

In [ ]:
X_tr, X_te, y_tr, y_te = skm.train_test_split(
    mensagens["texto"].values, y, test_size=0.3,
    random_state=2026, stratify=...)                            # (a)

vetorizador = CountVectorizer()
Xc = vetorizador.fit_transform(...)                          # (b) ajuste SO no treino

n_linhas, n_termos = Xc.shape
densidade = ...              # (c)

print(f"matriz: {Xc.shape}   nao-zeros: {Xc.nnz}   densidade: {densidade:.4f}%")
print(f"memoria esparsa: {(Xc.data.nbytes + Xc.indices.nbytes + Xc.indptr.nbytes) / 1e6:.2f} MB")
print(f"memoria densa:   {n_linhas * n_termos * 8 / 1e6:.1f} MB")

Veja quais termos dominam a matriz, e quantos aparecem uma única vez.

In [ ]:
contagens = np.asarray(Xc.sum(axis=0)).ravel()
vocabulario = vetorizador.get_feature_names_out()
ordem = ...                                # (a) do mais frequente ao menos

print("10 termos mais frequentes:")
print([(vocabulario[i], int(contagens[i])) for i in ordem[:10]])

# em quantos DOCUMENTOS cada termo aparece (nao quantas vezes ao todo)
n_docs = np.asarray((Xc > 0).sum(axis=0)).ravel()
raros = ...                              # (b) termos em um unico documento
print(f"\ntermos que aparecem em 1 documento so: {raros} de {n_termos} "
      f"({100 * raros / n_termos:.1f}%)")

---
## Exercício 2 — `min_df` como regularização

Se metade do vocabulário aparece uma vez só, cortar essa cauda deveria ajudar.
Meça quanto ela pesa.

In [ ]:
for m in (1, 2, 5, 10):
    v = CountVectorizer(min_df=...).fit(X_tr)                   # (a)
    print(f"min_df={m:2d}: {len(v.get_feature_names_out()):5d} termos")

---
## Exercício 3 — seis combinações

Duas representações (contagens e TF-IDF) e três classificadores lineares. Meça
acurácia, $F_1$ e — o que mais importa num filtro de spam — os falsos positivos e
falsos negativos separadamente.

In [ ]:
representacoes = [("contagens", ...),           # (a)
                  ("TF-IDF",    ...)]           # (b)

classificadores = [("MultinomialNB", MultinomialNB()),
                   ("logistica",     LogisticRegression(max_iter=2000)),
                   ("LinearSVC",     LinearSVC(dual=True, max_iter=5000))]

for nome_rep, vetor in representacoes:
    for nome_clf, modelo in classificadores:
        tubo = Pipeline([("vetor", vetor), ("clf", modelo)]).fit(X_tr, y_tr)
        pred = tubo.predict(X_te)
        vn, fp, fn, vp = ...  # (c)
        print(f"{nome_rep:10s} {nome_clf:14s} acuracia {tubo.score(X_te, y_te):.4f}"
              f"  F1 {f1_score(y_te, pred):.4f}   FP {fp:3d}  FN {fn:3d}")

> **Sua vez.** O `TF-IDF + MultinomialNB` tem **zero** falsos positivos. Isso o
> torna a melhor escolha para um filtro de spam? Olhe a coluna FN antes de
> responder, e pense em qual seria o desempenho de um classificador que responde
> sempre "não é spam".

---
## Exercício 4 — o que o modelo aprendeu

Um modelo linear sobre texto é diretamente legível: cada coluna é uma palavra, e
o coeficiente dela diz o quanto ela empurra para spam ou para não-spam.

In [ ]:
tubo = Pipeline([("vetor", TfidfVectorizer()),
                 ("clf", LogisticRegression(max_iter=2000))]).fit(X_tr, y_tr)

palavras = tubo.named_steps["vetor"].get_feature_names_out()
coef = tubo.named_steps["clf"].coef_[...]                       # (a) uma linha so: problema binario

ordem = ...                                      # (b) do mais negativo ao mais positivo

print("mais indicativas de SPAM:")
print([(palavras[i], round(coef[i], 3)) for i in ordem[...][:10]])    # (c)
print("\nmais indicativas de NAO-SPAM:")
print([(palavras[i], round(coef[i], 3)) for i in ordem[:10]])

> **Sua vez.** Monte a busca completa — `ngram_range` em `[(1,1), (1,2)]`,
> `min_df` em `[1, 2, 5]` e o `C` da logística — num único `GridSearchCV` sobre o
> `Pipeline`, com `scoring="f1"`. Compare o resultado com as seis linhas do
> Exercício 3.

---
## O que ficou

| Exercício | O que você mediu |
|---|---|
| 1 | densidade de 0,18%: o formato esparso ocupa **355×** menos memória |
| 1 | 53,9% dos termos aparecem em **um único** documento |
| 2 | exigir 2 documentos em vez de 1 corta o vocabulário pela metade |
| 3 | o TF-IDF **piora** o naive Bayes (0,9856 → 0,9605) e a logística (0,9815 → 0,9653) |
| 3 | o melhor por falsos positivos é `contagens + LinearSVC` (FP = 1), que não é o melhor em acurácia |
| 4 | dois dos preditores mais fortes de não-spam, `gt` e `lt`, são artefato de codificação |

---

## Encerramento

Esta lista fecha o curso, e vale olhar para trás pelo que as catorze listas
mediram, não pelo que os métodos prometem.

O erro de treino é otimista, e o otimismo tem fórmula. A validação cruzada
estima risco, e quem escolheu não pode reportar. Cotas superiores não atingidas
não são cotas erradas. O critério que separa vazamento grave de leve é olhar o
$Y$, não aprender dos dados. Ordenar bem não é estimar bem. Um aviso verdadeiro
("o naive Bayes é mal calibrado") pode ser condicional a uma hipótese que os seus
dados não satisfazem — e conferir custou uma linha.

Em todas essas vezes, o que resolveu foi a mesma coisa: **medir, em vez de
repetir.**